<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [2]:
# Install spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
import findspark
findspark.init()

In [4]:
# Create the SparkSession
spark = SparkSession.builder \
    .appName("Spark MLOps") \
    .getOrCreate()

In [ ]:
# Download The search term dataset from the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [5]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (169.45.118.108:443)
wget: can't open 'searchterms.csv': File exists


In [6]:
# Load the csv into a spark dataframe
df = spark.read.csv("searchterms.csv", header=True, inferSchema=True)


In [ ]:
# Print the number of rows and columns


In [7]:
# Number of rows
num_rows = df.count()

# Number of columns
num_cols = len(df.columns)

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")


Number of rows: 10000
Number of columns: 4


In [8]:
# Print the top 5 rows
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



In [ ]:
# Find out the datatype of the column searchterm?


In [9]:
df.schema['searchterm'].dataType

StringType()

In [ ]:
# How many times was the term `gaming laptop` searched?


In [10]:
df.filter("searchterm = 'gaming laptop'").count()


499

In [ ]:
# Print the top 5 most frequently used search terms?


In [11]:
# Using ascending=False inside sort
df.groupBy("searchterm") \
  .count() \
  .sort("count", ascending=False) \
  .show(5)


+-------------+-----+
|   searchterm|count|
+-------------+-----+
|mobile 6 inch| 2312|
|    mobile 5g| 2301|
|mobile latest| 1327|
|       laptop|  935|
|  tablet wifi|  896|
+-------------+-----+
only showing top 5 rows



In [12]:
# The pretrained sales forecasting model is available at  the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz


Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (198.23.119.245:443)
saving to 'model.tar.gz'
model.tar.gz         100% |********************************|  1490  0:00:00 ETA
'model.tar.gz' saved


In [15]:
!tar -xzf model.tar.gz


In [16]:
from pyspark.ml.regression import LinearRegressionModel

# Spark automatically reads the metadata and data subfolders
model = LinearRegressionModel.load("sales_prediction.model")


In [ ]:
# Using the sales forecast model, predict the sales for the year of 2023.


In [17]:
from pyspark.ml.feature import VectorAssembler

# 1. Create a DataFrame with the year 2023
data = [(2023,)]
df_2023 = spark.createDataFrame(data, ["year"])

# 2. Assemble the 'year' column into a feature vector
assembler = VectorAssembler(inputCols=["year"], outputCol="features")
features_df = assembler.transform(df_2023)

# 3. Predict sales using the loaded model
predictions = model.transform(features_df)

# 4. Display the prediction
predictions.select("year", "prediction").show()


+----+------------------+
|year|        prediction|
+----+------------------+
|2023|175.16564294006457|
+----+------------------+

